# Ingestion Pipeline
- second nootbook to Ingestion json to FAISS vector store

### **egyptian-civil-code.json** -> **Document objects** -> **Embeddings** -> **FAISS Vector Store** -> **Save locally**

### Install

In [71]:
%pip install sentence-transformers faiss-cpu
%pip install langchain langchain-community 
%pip install -U langchain-huggingface sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import the dependencies

In [72]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

### Load JSON file

In [73]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        articles = json.load(f)
    return articles

arabic_articles = load_json("../data/egyptian-civil-code-ar.json")
english_articles = load_json("../data/egyptian-civil-code-en.json")

print(f"✅ Loaded {len(arabic_articles)} مادة")
print(f"✅ Loaded {len(english_articles)} articles")


✅ Loaded 997 مادة
✅ Loaded 1070 articles


### convert json to document

In [74]:
def convert_json_to_docs(articles,lang):
    docs_ar = []
    doc_en = []

    if lang == "ar":
        for article in articles:
            combined_text = (
                f"مادة {article['article_number']}\n"
                f"{article['text']}"
            )

            doc = Document(
                page_content=combined_text,
                metadata={
                    "article_number": article['article_number'],
                    'is_repealed':    article['is_repealed'],
                    "source":         f"Article {article['article_number']}"
                }
            )

            docs_ar.append(doc)

        return docs_ar
    else:
        for article in articles:
            combined_text = (
                f"Article {article['article_number']}\n"
                f"{article['text']}"
            )

            doc = Document(
                page_content=combined_text,
                metadata={
                    "article_number": article['article_number'],
                    'is_repealed':    article['is_repealed'],
                    "source":         f"Article {article['article_number']}"
                }
            )

            doc_en.append(doc)

        return doc_en

docs_ar = convert_json_to_docs(arabic_articles,lang='ar')
docs_en = convert_json_to_docs(english_articles, lang="en")

print(f"✅ Created {len(docs_ar)} Arabic documents")
print(f"✅ Created {len(docs_en)} English documents \n")

print(f"\n=== Sample Arabic Document ===")
print(docs_ar[5].page_content)
print(f"Metadata: {docs_ar[5].metadata} \n")

print(f"\n=== Sample Arabic Document ===")
print(docs_ar[5].page_content)
print(f"Metadata: {docs_ar[5].metadata}")
    

✅ Created 997 Arabic documents
✅ Created 1070 English documents 


=== Sample Arabic Document ===
مادة 6
(١ (النصوص المتعلقة بالأهلية تسري علي جميع الأشخاص الذين تنطبق عليهم الشروط المقررة في هذه النصوص.
Metadata: {'article_number': 6, 'is_repealed': False, 'source': 'Article 6'} 


=== Sample Arabic Document ===
مادة 6
(١ (النصوص المتعلقة بالأهلية تسري علي جميع الأشخاص الذين تنطبق عليهم الشروط المقررة في هذه النصوص.
Metadata: {'article_number': 6, 'is_repealed': False, 'source': 'Article 6'}


### Load Embedding model

In [ ]:
def load_embeddings(model_name):
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings = load_embeddings("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding model loaded")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2068.76it/s]


✅ Embedding model loaded


### crreate FAISS Vector Store

In [77]:
print("⏳ Creating vector store...")

vector_store_ar = FAISS.from_documents(docs_ar, embeddings)
print("✅ arabic Vector store created")

vector_store_en = FAISS.from_documents(docs_en, embeddings)
print("✅ english Vector store created")


⏳ Creating vector store...
✅ arabic Vector store created
✅ english Vector store created


### Save Vector Store

In [79]:
ar_vector_store_path = Path("../data/vector_store_ar")
vector_store_ar.save_local(str(ar_vector_store_path))
print(f"✅ arabic Vector store saved to {ar_vector_store_path}")

en_vector_store_path = Path("../data/vector_store_en")
vector_store_en.save_local(str(en_vector_store_path))
print(f"✅ english Vector store saved to {en_vector_store_path}")

✅ arabic Vector store saved to ..\data\vector_store_ar
✅ english Vector store saved to ..\data\vector_store_en


### Load and test

In [81]:
def load_vector_store(path, embeddings):
    vector_store = FAISS.load_local(
        path, 
        embeddings,
        allow_dangerous_deserialization=True
    )
    return vector_store


ar_vector_store_loaded = load_vector_store(ar_vector_store_path, embeddings)

en_vector_store_loaded = load_vector_store(en_vector_store_path, embeddings)


question_ar = "ما هي شروط صحة العقد؟"
question_en = "What are the conditions of a valid contract?"

results_en = en_vector_store_loaded.similarity_search(question_en, k=3)
print(f"\n=== English Query: {question_en} ===")
print("========= Top 3 Results ==========")

for i, result in enumerate(results_en):
    print(f"\nResult {i+1}:")
    print(result.page_content)
    print(f"Metadata: {result.metadata}")


print("\n\n")

results_ar = ar_vector_store_loaded.similarity_search(question_ar, k=3)
print(f"\n=== Arabic Query: {question_ar} ===")
print("========= Top 3 Results ==========")

for i, result in enumerate(results_ar):
    print(f"\nResult {i+1}:")
    print(result.page_content)
    print(f"Metadata: {result.metadata}")



=== English Query: What are the conditions of a valid contract? ===
========= Top 3 Results ==========

Result 1:
Article 743
Subject to the requirements of the law as to the form of contracts for gifts, a contract providing for an annuity is valid only if made in writing.
Metadata: {'article_number': 743, 'is_repealed': False, 'source': 'Article 743'}

Result 2:
Article 144
When a void or voidable contract contains the elements of another contract, the contract will be deemed to be valid to the extent of the other contract, if it appears that the parties intended to conclude such another contract.
Metadata: {'article_number': 144, 'is_repealed': False, 'source': 'Article 144'}

Result 3:
Article 101
An agreement by which the two parties, or one of them, promise to enter into a particular contract in the future, is only binding if all the essential points of the contract envisaged and the time when the contract should be concluded are stated. When the law provides that a contract shal